# 从零实现 GAT：边注意力、目标分段 Softmax 与多头网络

本 Notebook 只用 PyTorch 基础张量和 `nn.Module` 手写 edge attention、按目标节点的 segment softmax、`GATLayer` 与两层多头 `GATNet`；不使用 PyG、DGL、`torch_geometric` 或现成 GNN 层。

链路包括 tenant 前置过滤、邻接 mask、自环去重、数值稳定 softmax、参数量/shape、mask 半监督训练、梯度、注意力和为 1、标签与 tenant 泄漏反例、state/artifact 指纹和推理合同。数据完全虚构、CPU 离线且固定种子。

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

from dataclasses import dataclass
import hashlib
import json
import math
import random
import time

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED=2701
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE=torch.device("cpu")
DTYPE=torch.float32

def canonical_fingerprint(payload)->str:
    raw=json.dumps(payload,ensure_ascii=False,sort_keys=True,separators=(",",":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]

assert DEVICE.type=="cpu" and torch.get_num_threads()==1
assert torch.initial_seed()==SEED
assert not any(name in globals() for name in ("torch_geometric","dgl"))

## 1. 受控图与 tenant 合同

tenant-a 有 18 个节点、两类各 9 个；同类环和二跳边占主导，另有少量跨类边。tenant-b 的极端特征以及一条恶意跨 tenant 边用于验证隔离。注意力不是先在全图计算再隐藏结果：softmax 的分母会因不可见邻居改变，因此 tenant/ACL 必须在 edge mask 和自环生成之前完成。

In [ ]:
@dataclass(frozen=True)
class AuthContext:
    tenant:str
    scopes:frozenset[str]
    principal:str
    def require(self,scope:str)->None:
        if scope not in self.scopes:
            raise PermissionError(f"缺少 scope: {scope}")

auth_a=AuthContext("tenant-a",frozenset({"graph:read","model:predict"}),"alice")
node_ids_all=[f"tenant-a:n{i:02d}" for i in range(18)]+["tenant-b:x0","tenant-b:x1"]
tenant_all=["tenant-a"]*18+["tenant-b"]*2
generator=torch.Generator().manual_seed(SEED)
base0,base1=torch.tensor([1.15,0.2,0.35,1.0]),torch.tensor([0.2,1.15,0.75,1.0])
X_a=torch.stack([(base0 if i<9 else base1)+0.23*torch.randn(4,generator=generator)*torch.tensor([1,1,1,0]) for i in range(18)])
X_all=torch.cat([X_a,torch.tensor([[100.,100.,100.,1.],[-100.,-100.,-100.,1.]])])
y_all=torch.tensor([0]*9+[1]*9+[-1,-1],dtype=torch.long)

pair_set=set()
for offset in (0,9):
    for j in range(9):
        for step in (1,2):
            pair_set.add(tuple(sorted((offset+j,offset+(j+step)%9))))
raw_pairs=sorted(pair_set|{(0,9),(4,13),(18,19),(0,18)})

def authorize(auth:AuthContext):
    auth.require("graph:read")
    visible=[i for i,t in enumerate(tenant_all) if t==auth.tenant]
    remap={old:new for new,old in enumerate(visible)}
    pairs=[(remap[u],remap[v]) for u,v in raw_pairs if u in remap and v in remap and tenant_all[u]==tenant_all[v]==auth.tenant]
    return [node_ids_all[i] for i in visible],X_all[visible].clone(),y_all[visible].clone(),sorted(pairs)

node_ids,X,y,undirected_pairs=authorize(auth_a)
assert X.shape==(18,4) and y.shape==(18,)
assert len(undirected_pairs)==38
assert set(y.tolist())=={0,1}
assert all(n.startswith("tenant-a:") for n in node_ids)

## 2. Mask 与半监督边界

每类 4 个 train、2 个 validation、3 个 test。所有授权节点的无标签特征与边参与 transductive attention，只有 train 标签进入交叉熵；validation 选择 checkpoint，test 冻结后只打开一次。GAT 可用于 inductive 图，但本 Notebook 的基础评估明确是 transductive，不能因模型名称而改写协议。

In [ ]:
train_mask=torch.zeros(18,dtype=torch.bool)
val_mask=torch.zeros(18,dtype=torch.bool)
test_mask=torch.zeros(18,dtype=torch.bool)
for cls in (0,1):
    idx=torch.where(y==cls)[0]
    train_mask[idx[:4]]=True; val_mask[idx[4:6]]=True; test_mask[idx[6:]]=True
coverage=train_mask.to(torch.int8)+val_mask.to(torch.int8)+test_mask.to(torch.int8)
assert (int(train_mask.sum()),int(val_mask.sum()),int(test_mask.sum()))==(8,4,6)
assert torch.equal(coverage,torch.ones_like(coverage))
assert set(y[train_mask].tolist())==set(y[val_mask].tolist())==set(y[test_mask].tolist())=={0,1}
assert not torch.any(train_mask&val_mask) and not torch.any(train_mask&test_mask)

## 3. 邻接 mask 与自环

约定 `source→target`：target 从 source 接收消息。无向边展开为两个方向；`with_exact_self_loops` 先去重全部边，再为每个节点恰好加入一个自环。GAT 只能在这些边上算 attention，不能为非邻接节点创建一个大负数后忘记彻底 mask。自环确保每个目标至少有一个入边，softmax 分母非空。

In [ ]:
def directed_edges(num_nodes:int,pairs:list[tuple[int,int]])->torch.Tensor:
    directed=[]
    for u,v in pairs:
        if u==v or not (0<=u<num_nodes and 0<=v<num_nodes):
            raise ValueError("边端点非法或输入含自环")
        directed.extend([(u,v),(v,u)])
    directed=sorted(set(directed),key=lambda p:(p[1],p[0]))
    return torch.tensor(directed,dtype=torch.long).T.contiguous()

def with_exact_self_loops(edges:torch.Tensor,num_nodes:int)->torch.Tensor:
    if edges.ndim!=2 or edges.shape[0]!=2:
        raise ValueError("edge_index shape 必须为 (2,E)")
    pairs={(int(s),int(t)) for s,t in edges.T.tolist() if int(s)!=int(t)}
    if any(s<0 or t<0 or s>=num_nodes or t>=num_nodes for s,t in pairs):
        raise ValueError("edge_index 越界")
    pairs.update((i,i) for i in range(num_nodes))
    ordered=sorted(pairs,key=lambda p:(p[1],p[0]))
    return torch.tensor(ordered,dtype=torch.long).T.contiguous()

base_edges=directed_edges(18,undirected_pairs)
masked_edges=with_exact_self_loops(base_edges,18)
loop_mask=masked_edges[0]==masked_edges[1]
loops_per_target=torch.bincount(masked_edges[1,loop_mask],minlength=18)
assert base_edges.shape==(2,76) and masked_edges.shape==(2,94)
assert torch.equal(loops_per_target,torch.ones(18,dtype=torch.long))
assert (0,5) not in set(map(tuple,masked_edges.T.tolist()))
assert all(int((masked_edges[1]==i).sum())>0 for i in range(18))

## 4. 按目标节点的 segment softmax

每条边、每个 head 有一个 logit (e_{s\to t}^{(h)})。归一化必须只在相同 target 的入边集合中进行：

[
\alpha_{s\to t}^{(h)}=
\frac{\exp(e_{s\to t}^{(h)}-m_t^{(h)})}
{\sum_{k\in N(t)\cup\{t\}}\exp(e_{k\to t}^{(h)}-m_t^{(h)})}
]

减去 segment 最大值避免溢出。对每个存在入边的 target/head，attention 之和应为 1。

In [ ]:
def target_segment_softmax(scores:torch.Tensor,target:torch.Tensor,num_nodes:int)->torch.Tensor:
    if scores.ndim!=2 or target.ndim!=1 or scores.shape[0]!=target.numel():
        raise ValueError("scores/target shape 不匹配")
    if target.numel() and (int(target.min())<0 or int(target.max())>=num_nodes):
        raise ValueError("target 越界")
    alpha=torch.zeros_like(scores)
    for node in range(num_nodes):
        mask=target==node
        if mask.any():
            local=scores[mask]
            shifted=local-local.max(dim=0,keepdim=True).values
            exp=shifted.exp()
            alpha[mask]=exp/exp.sum(dim=0,keepdim=True)
    return alpha

fixture_scores=torch.tensor([[1000.,-2.],[999.,-1.],[4.,7.],[4.,5.]])
fixture_target=torch.tensor([0,0,1,1])
fixture_alpha=target_segment_softmax(fixture_scores,fixture_target,2)
for node in (0,1):
    assert torch.allclose(fixture_alpha[fixture_target==node].sum(0),torch.ones(2),atol=1e-7)
np_ref=np.exp(np.array([1000.,999.])-1000.); np_ref=np_ref/np_ref.sum()
assert np.allclose(fixture_alpha[:2,0].numpy(),np_ref,atol=1e-7)
assert torch.isfinite(fixture_alpha).all() and torch.all(fixture_alpha>0)

## 5. 手写多头 `GATLayer`

每个 head 有 (W^{(h)}\in\mathbb{R}^{F_{in}\times F_{out}})、(a_s^{(h)},a_t^{(h)}\in\mathbb{R}^{F_{out}})。变换后：

[
e_{s\to t}^{(h)}=\operatorname{LeakyReLU}
((a_s^{(h)})^TWh_s+(a_t^{(h)})^TWh_t)
]

随后 target-segment softmax，并按 target 累加 (alpha Wh_s)。concat 模式输出 `(N,H*F_out)`，mean 模式输出 `(N,F_out)`。

主数据图把无向边展开成双向边，单靠它无法充分锁定 `source→target` 落点；而第二层只有一个 head，`concat=False` 也退化为恒等。下面增加固定权重的非对称 `0→1` fixture，并令两个 head 使用 1×/3× 投影。预期输出可精确手算，同时验证消息方向和真正的多头 mean。


In [ ]:
class GATLayer(nn.Module):
    def __init__(self,in_features:int,out_features:int,heads:int=1,concat:bool=True,
                 negative_slope:float=0.2,attention_dropout:float=0.0):
        super().__init__()
        if min(in_features,out_features,heads)<=0:
            raise ValueError("维度与 heads 必须为正")
        self.in_features,self.out_features,self.heads=in_features,out_features,heads
        self.concat,self.negative_slope=bool(concat),float(negative_slope)
        self.attention_dropout=float(attention_dropout)
        self.weight=nn.Parameter(torch.empty(heads,in_features,out_features))
        self.attn_source=nn.Parameter(torch.empty(heads,out_features))
        self.attn_target=nn.Parameter(torch.empty(heads,out_features))
        bias_size=heads*out_features if concat else out_features
        self.bias=nn.Parameter(torch.zeros(bias_size))
        nn.init.xavier_uniform_(self.weight)
        nn.init.xavier_uniform_(self.attn_source.unsqueeze(-1))
        nn.init.xavier_uniform_(self.attn_target.unsqueeze(-1))

    def forward(self,x:torch.Tensor,edges:torch.Tensor,return_attention:bool=False):
        if x.ndim!=2 or x.shape[1]!=self.in_features:
            raise ValueError("x shape 不匹配")
        used_edges=with_exact_self_loops(edges,x.shape[0]).to(x.device)
        source,target=used_edges
        transformed=torch.einsum("nf,hfo->nho",x,self.weight)
        score_source=(transformed[source]*self.attn_source.unsqueeze(0)).sum(-1)
        score_target=(transformed[target]*self.attn_target.unsqueeze(0)).sum(-1)
        scores=F.leaky_relu(score_source+score_target,negative_slope=self.negative_slope)
        alpha=target_segment_softmax(scores,target,x.shape[0])
        message_alpha=F.dropout(alpha,p=self.attention_dropout,training=self.training)
        messages=message_alpha.unsqueeze(-1)*transformed[source]
        aggregated=torch.zeros((x.shape[0],self.heads,self.out_features),dtype=x.dtype,device=x.device)
        aggregated.index_add_(0,target,messages)
        out=aggregated.reshape(x.shape[0],-1) if self.concat else aggregated.mean(dim=1)
        out=out+self.bias
        if not torch.isfinite(out).all():
            raise ValueError("GATLayer 产生非有限值")
        return (out,alpha,used_edges) if return_attention else out

probe_layer=GATLayer(4,3,heads=2,concat=True)
probe,probe_alpha,probe_edges=probe_layer(X,base_edges,return_attention=True)
assert probe.shape==(18,6) and probe_alpha.shape==(94,2)
assert sum(p.numel() for p in probe_layer.parameters())==2*4*3+2*2*3+2*3
assert probe_edges.shape==masked_edges.shape
assert torch.isfinite(probe).all()
# 非对称有向 oracle：只有 0→1（另由层内部恰好加入 0→0、1→1）。
oracle_layer27 = GATLayer(1, 1, heads=2, concat=False, attention_dropout=0.0)
with torch.no_grad():
    oracle_layer27.weight.zero_()
    oracle_layer27.weight[0, 0, 0] = 1.0
    oracle_layer27.weight[1, 0, 0] = 3.0
    oracle_layer27.attn_source.zero_(); oracle_layer27.attn_target.zero_(); oracle_layer27.bias.zero_()
oracle_layer27.eval()
oracle_X27 = torch.tensor([[2.0], [10.0]])
oracle_edges27 = torch.tensor([[0], [1]], dtype=torch.long)  # source row, target row
oracle_out27, oracle_alpha27, oracle_used27 = oracle_layer27(
    oracle_X27, oracle_edges27, return_attention=True
)
oracle_edge_set27 = set(map(tuple, oracle_used27.T.tolist()))
assert oracle_edge_set27 == {(0, 0), (0, 1), (1, 1)}
assert (1, 0) not in oracle_edge_set27
assert torch.allclose(oracle_out27, torch.tensor([[4.0], [12.0]]), atol=1e-7)
assert torch.allclose(oracle_alpha27[oracle_used27[1] == 0], torch.ones(1, 2), atol=1e-7)
assert torch.allclose(oracle_alpha27[oracle_used27[1] == 1], torch.full((2, 2), 0.5), atol=1e-7)
assert oracle_layer27.heads == 2 and oracle_layer27.concat is False


## 6. 两层 `GATNet` 与参数量

第一层 2 个 head、每头 4 维并 concat，得到 8 维；ELU/Dropout 后，第二层 1 个 head 输出 2 类且不 concat。第一层参数 56，第二层 22，总计 78。返回的是 logits；attention 只在诊断模式返回，避免服务默认泄露完整邻接证据。

In [ ]:
class GATNet(nn.Module):
    def __init__(self,in_features:int,hidden_per_head:int,classes:int,heads:int=2,dropout:float=0.1):
        super().__init__()
        self.gat1=GATLayer(in_features,hidden_per_head,heads=heads,concat=True,attention_dropout=0.05)
        self.gat2=GATLayer(hidden_per_head*heads,classes,heads=1,concat=False,attention_dropout=0.0)
        self.dropout=float(dropout)

    def forward(self,x:torch.Tensor,edges:torch.Tensor,return_attention:bool=False):
        if return_attention:
            h,a1,e1=self.gat1(x,edges,return_attention=True)
            h=F.elu(h); h=F.dropout(h,p=self.dropout,training=self.training)
            logits,a2,e2=self.gat2(h,edges,return_attention=True)
            return logits,{"layer1":(a1,e1),"layer2":(a2,e2)}
        h=F.elu(self.gat1(x,edges))
        h=F.dropout(h,p=self.dropout,training=self.training)
        return self.gat2(h,edges)

model=GATNet(4,4,2,heads=2)
model.eval(); logits,attention=model(X,base_edges,return_attention=True)
assert logits.shape==(18,2)
assert sum(p.numel() for p in model.parameters())==78
assert attention["layer1"][0].shape==(94,2) and attention["layer2"][0].shape==(94,1)
assert torch.isfinite(logits).all()

## 7. Attention 不变量：每个 target/head 和为 1

该断言必须按 target 分组；对全图或 source 求和都不是 GAT 的归一化语义。还要确认 alpha 只对应 mask 后的边，自环恰好一次。训练态的 attention dropout 会让实际消息权重不再严格和为 1，因此此处在 `eval()` 下检查返回的 dropout 前 alpha。

In [ ]:
model.eval()
with torch.no_grad():
    _,attention=model(X,base_edges,return_attention=True)
for layer_name,(alpha,edges_used) in attention.items():
    targets=edges_used[1]
    for node in range(18):
        sums=alpha[targets==node].sum(dim=0)
        assert torch.allclose(sums,torch.ones_like(sums),atol=1e-6), (layer_name,node,sums)
    assert torch.all(alpha>=0) and torch.all(alpha<=1)
    assert torch.equal(torch.bincount(edges_used[1,edges_used[0]==edges_used[1]],minlength=18),torch.ones(18,dtype=torch.long))
assert set(map(tuple,attention["layer1"][1].T.tolist()))==set(map(tuple,masked_edges.T.tolist()))
assert (0,5) not in set(map(tuple,attention["layer1"][1].T.tolist()))

## 8. Train mask 训练与 validation checkpoint

每轮 full-batch forward 只在授权边上做 attention；交叉熵严格索引 `train_mask`。validation loss 只选择 checkpoint，不反传。保存 detached clone，恢复后才计算 test。小图上的 full-batch Python segment 循环用于清晰性，生产不能据此估算吞吐。

训练前保存参数副本与 eval-mode train loss；恢复最佳 checkpoint 后要求参数确实变化、train loss 明显下降、train-mask accuracy 达标。这样 `best_state`、有限 history 和非零梯度不再掩盖误删 `optimizer.step()` 的回归。


In [ ]:
torch.manual_seed(SEED)
model = GATNet(4, 4, 2, heads=2, dropout=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.035, weight_decay=5e-4)
initial_train_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
model.eval()
with torch.no_grad():
    initial_train_eval_loss = float(F.cross_entropy(model(X, base_edges)[train_mask], y[train_mask]))

best_val, best_epoch, best_state = math.inf, -1, None
history = []; started = time.perf_counter()
for epoch in range(120):
    model.train(); optimizer.zero_grad(set_to_none=True)
    logits = model(X, base_edges)
    loss = F.cross_entropy(logits[train_mask], y[train_mask])
    loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad():
        val_loss = float(F.cross_entropy(model(X, base_edges)[val_mask], y[val_mask]))
    history.append((float(loss.detach()), val_loss))
    if val_loss < best_val:
        best_val, best_epoch = val_loss, epoch
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
train_seconds = time.perf_counter() - started

assert best_state is not None and 0 <= best_epoch < 120
assert len(history) == 120 and np.isfinite(np.asarray(history)).all()
assert train_seconds < 20
model.load_state_dict(best_state, strict=True); model.eval()
with torch.no_grad():
    restored_train_logits = model(X, base_edges)
    restored_train_eval_loss = float(F.cross_entropy(restored_train_logits[train_mask], y[train_mask]))
    restored_train_accuracy = float((restored_train_logits[train_mask].argmax(1) == y[train_mask]).float().mean())
updated_parameter_keys = [key for key, value in model.state_dict().items()
                          if not torch.equal(value.detach(), initial_train_state[key])]
assert updated_parameter_keys, "optimizer 没有改变任何 state_dict 张量"
assert restored_train_eval_loss < initial_train_eval_loss * 0.75
assert restored_train_accuracy >= 0.90


## 9. 非零有限梯度

GAT 常见错误是把 alpha 转 NumPy、在 segment 操作中 detach，或用不可微覆盖导致 attention 参数无梯度。下面对恢复后的模型做一次 train-mask backward，要求权重、source attention、target attention和 bias 均有有限非零梯度；不执行 optimizer step，因此不会改变制品。

In [ ]:
model.train(); optimizer.zero_grad(set_to_none=True)
check_loss=F.cross_entropy(model(X,base_edges)[train_mask],y[train_mask])
check_loss.backward()
grad_norms={name:float(p.grad.norm()) for name,p in model.named_parameters()}
assert len(grad_norms)==8
assert all(math.isfinite(v) and v>0 for v in grad_norms.values())
assert any("attn_source" in name for name in grad_norms)
assert any("attn_target" in name for name in grad_norms)
model.eval()

## 10. Test 与标签泄漏反例

冻结 checkpoint 后只报告一次 test accuracy。为了验证损失边界，把所有 test 标签翻转；只要训练损失确实只索引 train mask，数值必须完全不变。这个反例只验证标签索引，不替代特征 event-time、预处理 fit split 与 lineage 审计。

In [ ]:
with torch.no_grad():
    final_logits=model(X,base_edges)
    final_pred=final_logits.argmax(1)
test_accuracy=float((final_pred[test_mask]==y[test_mask]).float().mean())
original_train_loss=F.cross_entropy(final_logits[train_mask],y[train_mask])
poisoned_y=y.clone(); poisoned_y[test_mask]=1-poisoned_y[test_mask]
poisoned_train_loss=F.cross_entropy(final_logits[train_mask],poisoned_y[train_mask])
assert 0<=test_accuracy<=1
assert torch.equal(y[train_mask],poisoned_y[train_mask])
assert torch.allclose(original_train_loss,poisoned_train_loss,atol=0,rtol=0)
assert not torch.equal(y[test_mask],poisoned_y[test_mask])

## 11. 查看 attention，但不要当作因果解释

下面仅列出第一层 head-0 对一个目标节点的最高入边权重。attention 是模型内部、特定层/head/参数化下的归一化系数；它不表示干预后的因果效应，也没有说明 source 特征的正负方向。生产展示还需权限过滤、稳定节点 ID、层/head 标识和模型版本。

In [ ]:
with torch.no_grad():
    _,final_attention=model(X,base_edges,return_attention=True)
alpha1,edges1=final_attention["layer1"]
target_node=0
incoming=torch.where(edges1[1]==target_node)[0]
ranked=sorted([(node_ids[int(edges1[0,i])],float(alpha1[i,0])) for i in incoming],key=lambda x:(-x[1],x[0]))
attention_sum=sum(score for _,score in ranked)
assert len(ranked)==int((edges1[1]==target_node).sum())
assert abs(attention_sum-1.0)<1e-6
assert ranked[0][1]>=ranked[-1][1]>=0
assert all(node.startswith("tenant-a:") for node,_ in ranked)

## 12. tenant 泄漏：softmax 分母也会泄漏

把 tenant-b 极端节点通过恶意边连到 node 0 后，全图 attention 的候选集合、分母和消息都会改变。下面用同一已训练模型对比安全图与未过滤原始图；差异证明“最后裁掉 tenant-b 输出”不够。

In [ ]:
raw_edges=directed_edges(20,raw_pairs)
model.eval()
with torch.no_grad():
    safe_node0=model(X,base_edges)[0]
    unsafe_node0=model(X_all,raw_edges)[0]
assert not torch.allclose(safe_node0,unsafe_node0)
unsafe_with_loops=with_exact_self_loops(raw_edges,20)
assert (18,0) in set(map(tuple,unsafe_with_loops.T.tolist()))
assert (18,0) not in set(map(tuple,masked_edges.T.tolist()))
assert X.shape[0]<X_all.shape[0]

## 13. Artifact 与图/特征/state 绑定

GAT 对 edge mask 和节点顺序敏感，因此制品必须绑定授权图指纹、feature schema、自环策略、head 结构及 state_dict。state 指纹按 key、dtype、shape、原始字节计算。内容哈希可发现错配，但生产仍应使用签名制品、不可变注册表和受控加载。

除 schema 外，制品还绑定 `snapshot_id/as_of/node_order/shape/dtype/content_sha256` 组成的有序特征快照。验证函数从当前实际 `X` 重建描述；任一特征值变化都会被拒绝。architecture 也记录 concat、negative slope 和 dropout，避免相同权重被不同 forward 语义解释。


In [ ]:
FEATURE_SCHEMA = {"order": ["http_ratio","batch_ratio","cpu_norm","bias"],
                  "dtype": "float32", "source": "synthetic-v1", "fit_split": "not_applicable"}
FEATURE_SCHEMA_ID = canonical_fingerprint(FEATURE_SCHEMA)
FEATURE_SNAPSHOT_ID = "tenant-a-gat-features-2026-07-01T00:00:00Z"
FEATURE_SNAPSHOT_AS_OF = "2026-07-01T00:00:00Z"

def feature_snapshot_descriptor(ordered_node_ids, features, snapshot_id: str, as_of: str) -> dict:
    if features.ndim != 2 or len(ordered_node_ids) != features.shape[0]:
        raise ValueError("节点顺序与特征 shape 不匹配")
    if len(set(ordered_node_ids)) != len(ordered_node_ids) or not torch.isfinite(features).all():
        raise ValueError("节点 ID 必须唯一且特征必须有限")
    value = features.detach().cpu().contiguous()
    return {"snapshot_id": snapshot_id, "as_of": as_of,
            "node_order": list(ordered_node_ids), "shape": list(value.shape),
            "dtype": str(value.dtype),
            "content_sha256": hashlib.sha256(value.numpy().tobytes()).hexdigest()}

FEATURE_SNAPSHOT = feature_snapshot_descriptor(
    node_ids, X, FEATURE_SNAPSHOT_ID, FEATURE_SNAPSHOT_AS_OF
)
FEATURE_SNAPSHOT_FINGERPRINT = canonical_fingerprint(FEATURE_SNAPSHOT)
graph_payload = {
    "tenant": auth_a.tenant, "as_of": "2026-07-01T00:00:00Z", "nodes": node_ids,
    "edges": sorted([sorted((node_ids[u],node_ids[v])) for u,v in undirected_pairs]),
    "self_loop_policy": "exactly-one", "edge_direction": "source-to-target",
}
GRAPH_FINGERPRINT = canonical_fingerprint(graph_payload)

def state_dict_fingerprint(state) -> str:
    digest = hashlib.sha256()
    for key in sorted(state):
        tensor = state[key].detach().cpu().contiguous()
        digest.update(key.encode()); digest.update(str(tensor.dtype).encode())
        digest.update(json.dumps(list(tensor.shape)).encode()); digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()[:20]

artifact = {
    "model_type": "GATNet-from-scratch", "model_version": "gat-v2",
    "tenant": auth_a.tenant, "as_of": graph_payload["as_of"],
    "graph_fingerprint": GRAPH_FINGERPRINT, "feature_schema_id": FEATURE_SCHEMA_ID,
    "feature_snapshot_id": FEATURE_SNAPSHOT_ID,
    "feature_snapshot_fingerprint": FEATURE_SNAPSHOT_FINGERPRINT,
    "architecture": {"dims": [4,4,2], "heads": [2,1], "concat": [True,False],
                     "negative_slope": 0.2, "hidden_dropout": 0.1,
                     "attention_dropout": [0.05,0.0]},
    "self_loop_policy": "exactly-one",
    "state_dict_fingerprint": state_dict_fingerprint(model.state_dict()),
}
artifact["artifact_id"] = canonical_fingerprint(artifact)

def validate_artifact(value, state, current_feature_snapshot) -> bool:
    unsigned = {k:v for k,v in value.items() if k != "artifact_id"}
    if canonical_fingerprint(unsigned) != value.get("artifact_id"):
        raise ValueError("artifact hash 不匹配")
    if (value.get("graph_fingerprint") != GRAPH_FINGERPRINT or
            value.get("feature_schema_id") != FEATURE_SCHEMA_ID):
        raise ValueError("图或特征 schema 不匹配")
    current_feature_fp = canonical_fingerprint(current_feature_snapshot)
    if (value.get("feature_snapshot_id") != current_feature_snapshot.get("snapshot_id") or
            value.get("feature_snapshot_fingerprint") != current_feature_fp):
        raise ValueError("有序特征快照不匹配")
    if (value.get("self_loop_policy") != "exactly-one" or
            value.get("state_dict_fingerprint") != state_dict_fingerprint(state)):
        raise ValueError("自环或 state_dict 合同不匹配")
    return True

assert validate_artifact(artifact, model.state_dict(), FEATURE_SNAPSHOT)
assert len(artifact["artifact_id"]) == 20
for field, bad in (("graph_fingerprint","bad"),
                   ("feature_schema_id","bad"),
                   ("feature_snapshot_fingerprint","bad"),
                   ("state_dict_fingerprint","bad")):
    forged = dict(artifact); forged[field] = bad
    forged["artifact_id"] = canonical_fingerprint({k:v for k,v in forged.items() if k != "artifact_id"})
    try:
        validate_artifact(forged, model.state_dict(), FEATURE_SNAPSHOT)
        raise AssertionError(f"伪造 {field} 未拒绝")
    except ValueError:
        pass

changed_X27 = X.clone(); changed_X27[0, 0] += 25.0
changed_feature_snapshot27 = feature_snapshot_descriptor(
    node_ids, changed_X27, FEATURE_SNAPSHOT_ID, FEATURE_SNAPSHOT_AS_OF
)
assert canonical_fingerprint(changed_feature_snapshot27) != FEATURE_SNAPSHOT_FINGERPRINT
try:
    validate_artifact(artifact, model.state_dict(), changed_feature_snapshot27)
    raise AssertionError("特征内容改变后仍通过 artifact 校验")
except ValueError as exc:
    assert "特征快照" in str(exc)


## 14. 推理合同

客户端只提交稳定节点 ID，不能提交 edge mask、覆盖 tenant 或指定另一个 graph fingerprint。服务从内部授权快照加载 `X/base_edges/model/artifact`，验证 bundle 后整图前向，再裁出请求节点。默认只返回概率和 trace；若开放 attention 诊断，必须单独 scope、限量并经过邻居 ACL。

服务从当前实际 `X` 重算有序特征快照并校验，trace 同时携带 snapshot ID、as-of 与内容指纹。由此，同一 graph fingerprint 下的特征漂移不能静默复用旧制品。


In [ ]:
def predict_known(auth: AuthContext, requested: list[str], model, artifact):
    auth.require("model:predict")
    if auth.tenant != artifact.get("tenant"):
        raise PermissionError("tenant 与制品不匹配")
    current_feature_snapshot = feature_snapshot_descriptor(
        node_ids, X, FEATURE_SNAPSHOT_ID, FEATURE_SNAPSHOT_AS_OF
    )
    validate_artifact(artifact, model.state_dict(), current_feature_snapshot)
    index = {node:i for i,node in enumerate(node_ids)}
    if not requested or any(node not in index for node in requested):
        raise PermissionError("未知或跨 tenant 节点")
    model.eval()
    with torch.no_grad():
        probs = model(X, base_edges).softmax(1)
    return {"predictions": [{"node_id":node, "probabilities":probs[index[node]].tolist()}
                            for node in requested],
            "trace": {"artifact_id":artifact["artifact_id"],
                      "graph_fingerprint":GRAPH_FINGERPRINT,
                      "feature_schema_id":FEATURE_SCHEMA_ID,
                      "feature_snapshot_id":current_feature_snapshot["snapshot_id"],
                      "feature_snapshot_as_of":current_feature_snapshot["as_of"],
                      "feature_snapshot_fingerprint":canonical_fingerprint(current_feature_snapshot),
                      "attention_exposed":False}}

served = predict_known(auth_a, node_ids[:2], model, artifact)
assert len(served["predictions"]) == 2
assert served["trace"]["artifact_id"] == artifact["artifact_id"]
assert served["trace"]["feature_snapshot_fingerprint"] == FEATURE_SNAPSHOT_FINGERPRINT
assert all(abs(sum(row["probabilities"])-1) < 1e-6 for row in served["predictions"])
try:
    predict_known(auth_a, ["tenant-b:x0"], model, artifact)
    raise AssertionError("越权节点未拒绝")
except PermissionError:
    pass


## 15. 复杂度、数值与生产替换点

本实现每层投影约 (O(NHF_{in}F_{out}))，边打分/聚合约 (O(EHF_{out}))。Python 按节点循环 segment softmax 仅适合公式教学；生产应使用融合 scatter/segment kernel、稀疏批处理、邻居采样、混合精度下的稳定 softmax、显存预算和超时降级。不要构造 (N\times N) 全连接 attention mask。

上线需报告 0-hop/GCN/GraphSAGE 基线、多 seed 置信区间、head/度数/时间切片、attention 熵与饱和、标签延迟、动态图版本、tenant 越权、模型签名、灰度与回滚。attention 可视化不是因果解释，受控小图高分不能外推。

In [ ]:
with torch.no_grad():
    _, checked_attention = model(X, base_edges, return_attention=True)
for alpha, edges in checked_attention.values():
    for node in range(18):
        assert torch.allclose(alpha[edges[1] == node].sum(0), torch.ones(alpha.shape[1]), atol=1e-6)
assert isinstance(model.gat1, GATLayer) and isinstance(model.gat2, GATLayer)
assert model.gat1.attn_source.grad is not None and model.gat2.attn_target.grad is not None
assert state_dict_fingerprint(model.state_dict()) == artifact["state_dict_fingerprint"]
assert GRAPH_FINGERPRINT == canonical_fingerprint(graph_payload)
assert updated_parameter_keys and restored_train_eval_loss < initial_train_eval_loss
assert artifact["feature_snapshot_fingerprint"] == canonical_fingerprint(FEATURE_SNAPSHOT)
assert torch.allclose(oracle_out27, torch.tensor([[4.0],[12.0]]), atol=1e-7)
assert train_seconds < 20 and validate_artifact(artifact, model.state_dict(), FEATURE_SNAPSHOT)
print({"status":"PASS", "model":"GAT-from-scratch", "best_epoch":best_epoch,
       "test_accuracy":round(test_accuracy,3),
       "train_loss_before_after":[round(initial_train_eval_loss,4),round(restored_train_eval_loss,4)],
       "params":sum(p.numel() for p in model.parameters()), "seconds":round(train_seconds,3)})


## 16. 原始与官方资料

- Veličković et al., *Graph Attention Networks*：https://arxiv.org/abs/1710.10903
- PyTorch 官方 `nn.Module` 文档：https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html
- PyTorch 官方 Module/State 文档：https://docs.pytorch.org/docs/stable/notes/modules.html
- PyTorch 官方初始化 API：https://docs.pytorch.org/docs/stable/nn.init.html

原论文用于 masked self-attention、多头与 concat/average 背景；目标分段实现、tenant 前置过滤、制品指纹、注意力暴露策略和生产失败边界是本 Notebook 的工程扩展。